In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime
import uuid


In [0]:
spark.sql('use catalog datacrunchbase_sql_azure_database_catalog')

In [0]:
spark.sql("create schema if not exists workspace.bronze_schema")

In [0]:
spark.sql("""
          CREATE TABLE IF NOT EXISTS workspace.bronze_schema.ingestion_control(
              layer STRING,
              table_name STRING,
              ts_col STRING,
              pk_col STRING,
              last_successfull_ts TIMESTAMP,
              last_successfull_pk STRING,
              last_run_id STRING,
              rows_written bigint,
              run_status string,
              updated_at timestamp
          )
          USING DELTA
          """)

In [0]:
table_config={
    "orders":{"pk_col":"order_id","ts_col":"updated_at"},
    "products":{"pk_col":"product_id","ts_col":"updated_at"},
    "payments":{"pk_col":"payment_id","ts_col":"processed_at"}
}

bronze_run_id=str(uuid.uuid4())
print(bronze_run_id)

In [0]:
from pyspark.sql import functions as F

def get_last_successful_watermark(table_name: str):
    ctrl = (
        spark.table("workspace.bronze_schema.ingestion_control")  # change if needed
        .filter(
            (F.col("layer") == "bronze") &
            (F.col("table_name") == table_name) &
            (F.col("run_status") == "success")
        )
        .orderBy(F.col("updated_at").desc())
        .limit(1)
    )

    rows = ctrl.collect()

    if not rows:
        return None, None

    return rows[0]["last_successfull_ts"], rows[0]["last_successfull_pk"]

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime

def upsert_bronze_control(table_name, ts_col, pk_col, last_ts, last_pk, rows_written, run_id):

    # Step 1: Create DataFrame (new record)
    control_df = spark.createDataFrame(
        [(
            "bronze",
            table_name,
            ts_col,
            pk_col,
            last_ts,
            str(last_pk) if last_pk is not None else None,
            run_id,
            int(rows_written),
            "success",
            datetime.now()
        )],
        [
            "layer",
            "table_name",
            "ts_col",
            "pk_col",
            "last_successfull_ts",
            "last_successfull_pk",
            "last_run_id",
            "rows_written",
            "run_status",
            "updated_at"
        ]
    )

    # Step 2: Reference Delta table (USE YOUR CATALOG)
    dt = DeltaTable.forName(
        spark,
        "workspace.bronze_schema.ingestion_control"   # change if needed
    )

    # Step 3: Merge (Upsert logic)
    (
        dt.alias("t")
        .merge(
            control_df.alias("s"),
            "t.table_name = s.table_name AND t.layer = s.layer"
        )
        .whenMatchedUpdate(set={
            "ts_col": "s.ts_col",
            "pk_col": "s.pk_col",
            "last_successfull_ts": "s.last_successfull_ts",
            "last_successfull_pk": "s.last_successfull_pk",
            "last_run_id": "s.last_run_id",
            "rows_written": "s.rows_written",
            "run_status": "s.run_status",
            "updated_at": "s.updated_at"
        })
        .whenNotMatchedInsertAll()
        .execute()
    )

In [0]:
for table_name, cfg in table_config.items():

    pk_col = cfg["pk_col"]
    ts_col = cfg["ts_col"]
# datacrunchbase_sql_azure_database_catalog
    source_table = f"datacrunchbase_sql_azure_database_catalog.dbo.{table_name}"
    target_table = f"workspace.bronze_schema.{table_name}_raw"

    last_successful_ts, last_successful_pk = get_last_successful_watermark(table_name)

    print(f"\n=== Processing {table_name} ===")
    print(f"Last successful ts: {last_successful_ts}")
    print(f"Last successful pk: {last_successful_pk}")

    source_df = spark.read.table(source_table)

    if last_successful_ts is None:
        rows_to_load = source_df
    else:
        rows_to_load = source_df.filter(
            (F.col(ts_col) > F.lit(last_successful_ts)) |
            (
                (F.col(ts_col) == F.lit(last_successful_ts)) &
                (F.col(pk_col).cast("long") > F.lit(int(last_successful_pk)))
            )
        )

    rows_to_load = (
        rows_to_load
        .withColumn("bronze_ingested_at", F.current_timestamp())
        .withColumn("bronze_run_id", F.lit(bronze_run_id))
        .withColumn("bronze_source_table", F.lit(source_table))
    )

    row_count = rows_to_load.count()

    if row_count == 0:
        print(f"No new rows for {table_name}.")
        upsert_bronze_control(
            table_name,
            ts_col,
            pk_col,
            last_successful_ts,
            last_successful_pk,
            row_count,
            bronze_run_id
        )
        continue

    rows_to_load.write.format("delta").mode("append").saveAsTable(target_table)

    max_ts = rows_to_load.agg(F.max(ts_col).alias("max_ts")).collect()[0]["max_ts"]

    max_pk = (
        rows_to_load
        .filter(F.col(ts_col) == F.lit(max_ts))
        .agg(F.max(F.col(pk_col).cast("long")).alias("max_pk"))
        .collect()[0]["max_pk"]
    )

    upsert_bronze_control(
        table_name,
        ts_col,
        pk_col,
        max_ts,
        max_pk,
        row_count,
        bronze_run_id
    )

    print(f"Wrote {row_count} rows to {target_table}")